### Phase 2 — Geometry + Linker Modeling
#### Objective
Evaluate whether linker descriptors improve CO₂ uptake predictions
beyond the four existing geometry features.
## Models
1. Geometry-only Random Forest
2. Geometry + Linker Random Forest

Use identical MOF rows and train/validation/test splits for both models
at all five pressures.

---

In [7]:
# Step 1: import the libraries and connect the notebook to the project.
# Import pandas for loading and working with the dataset
import pandas as pd
# Import NumPy for checking numerical descriptor values
import numpy as np
# Import tools for locating project files
from pathlib import Path
import sys
# Define the project folder
project_root = Path("/home/susan/mof-co2-adsorption")

# Allow this notebook to import functions from src
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import  linker descriptor function
from src.linker_descriptors import calculate_all_descriptors

# Confirm that the function comes from the script
print(calculate_all_descriptors.__module__)

src.linker_descriptors


In [8]:
# Locate the processed-data folder
processed_folder = project_root / "data" / "processed"

# Display CSV and pickle filenames
for file in sorted(processed_folder.iterdir()):
    if file.suffix in {".csv", ".pkl"}:
        print(file.name)

df_chem.csv
hmof_linker_descriptors.pkl
hmof_linker_extracted_final.csv
hmof_linker_extracted_final.pkl
rows_with_both_problems.csv


In [9]:
#Step 3: check whether the pickle file includes linker descriptors
# Load your saved linker dataset
df_chemistry = pd.read_pickle(
    processed_folder / "hmof_linker_descriptors.pkl"
)

# Check its size
print("Dataset shape:", df_chemistry.shape)

# Find descriptor columns such as linker_1_MolWt
descriptor_columns = [
    column for column in df_chemistry.columns
    if any(column.startswith(f"linker_{i}_") for i in range(1, 14))
]

print("Descriptor columns:", len(descriptor_columns))

Dataset shape: (25928, 222)
Descriptor columns: 195


In [10]:
# step 4 : check whether missing descriptors occur only where a linker is absent.
# Collect validation results for each linker position
checks = []

for i in range(1, 14):
    linker = f"linker_{i}"

    # Identify rows containing a linker SMILES
    present = (
        df_chemistry[linker].notna()
        & df_chemistry[linker].fillna("").str.strip().ne("")
    )

    # Select this linker's descriptor columns
    columns = [
        col for col in descriptor_columns
        if col.startswith(f"{linker}_")
    ]
    values = df_chemistry[columns]

    # Check for missing or infinite descriptor values
    invalid_values = values.isna() | values.isin([np.inf, -np.inf])

    # Summarize results for this linker position
    checks.append({
        "linker": linker,
        "descriptor_count": len(columns),
        "present_linkers": int(present.sum()),
        "absent_linkers": int((~present).sum()),
        "present_with_invalid_values": int(
            (present & invalid_values.any(axis=1)).sum()
        ),
        "absent_with_descriptor_values": int(
            (~present & values.notna().any(axis=1)).sum()
        ),
    })

# Display the validation table
pd.DataFrame(checks)

,linker,descriptor_count,present_linkers,absent_linkers,present_with_invalid_values,absent_with_descriptor_values
0,linker_1,15,25928,0,0,0
1,linker_2,15,24899,1029,0,0
2,linker_3,15,20414,5514,0,0
3,linker_4,15,1250,24678,0,0
4,linker_5,15,261,25667,0,0
5,linker_6,15,83,25845,0,0
6,linker_7,15,29,25899,0,0
7,linker_8,15,15,25913,0,0
8,linker_9,15,10,25918,0,0
9,linker_10,15,4,25924,0,0


In [11]:
# Step 5: identify the MOF row with the invalid linker_1 descriptor value
linker = "linker_1"
columns = [col for col in descriptor_columns if col.startswith(f"{linker}_")]

present = (
    df_chemistry[linker].notna()
    & df_chemistry[linker].fillna("").str.strip().ne("")
)
values = df_chemistry[columns]
invalid_values = values.isna() | values.isin([np.inf, -np.inf])

problem_mask = present & invalid_values.any(axis=1)
problem_rows = df_chemistry.loc[problem_mask, ["filename", linker] + columns]

print("Number of affected rows:", problem_mask.sum())
problem_rows

Number of affected rows: 0


,filename,linker_1,linker_1_MolWt,linker_1_TPSA,linker_1_NumHAcceptors,linker_1_NumHDonors,linker_1_NumAromaticRings,linker_1_MaxPartialCharge,linker_1_MinPartialCharge,linker_1_NOCount,linker_1_NumHeteroatoms,linker_1_LogP,linker_1_LabuteASA,linker_1_RotatableBonds,linker_1_FractionCSP3,linker_1_PEOE_VSA1,linker_1_PEOE_VSA2


---
### Descriptor Validation — Missing and Infinite Values

The project’s chemistry dataset contains 25,929 MOFs and 195 linker
descriptor columns: 15 descriptors for each of 13 linker positions.

The validation check showed that:

- All 13 linker positions have the expected 15 descriptor columns.
- Absent linker positions contain only missing descriptor values, as expected.
- One MOF has a linker present in `linker_1` but at least one missing
  (`NaN`) or infinite (`inf`) descriptor value.
- No such issues were found in occupied positions `linker_2`–`linker_13`.

The value `present_with_invalid_values = 1` counts one affected MOF row,
not the number of affected descriptors. It does not establish that the
linker itself is invalid.

The next step identifies the MOF, its linker SMILES, and the affected
descriptors before deciding how to handle the row.

---